# Lab 1 · Part A — Kaggle (Starter)

**Fork this notebook** and work through it next to `kaggle_assignment.md`.

Cells marked 🛠 **DO IN THE KAGGLE UI** are platform actions — do them in the sidebar/menus, then run the code cell below to confirm.

- **Model:** `mistralai/Mistral-7B-Instruct-v0.1`
- **Tool:** repeng (control vectors)
- **Goal:** operate Kaggle end-to-end. The ML is provided and trivial on purpose.

## A2 · Turn on the GPU
🛠 **Settings → Accelerator → GPU T4 ×2** (or P100). Then run:

In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

## A3 · Add Mistral-7B from Kaggle Models
🛠 Right sidebar → **Add Input → Models** → search *Mistral 7B Instruct* → add it. It mounts under `/kaggle/input/…`. Copy that path into `MODEL_PATH` below.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

# 🛠 EDIT: set this to the path shown after you add the Kaggle Models input.
#    (Verify the exact path on the live Kaggle Models catalog — it may differ.)
MODEL_PATH = "/kaggle/input/mistral/pytorch/7b-instruct-v0.1/1"

tok = AutoTokenizer.from_pretrained(MODEL_PATH)
m = AutoModelForCausalLM.from_pretrained(MODEL_PATH, torch_dtype=torch.float16).to("cuda:0")
ids = tok("[INST] Say hello in five words. [/INST]", return_tensors="pt").to(m.device)
print(tok.decode(m.generate(**ids, max_new_tokens=20)[0], skip_special_tokens=True))
del m; torch.cuda.empty_cache()   # free memory before the repeng section

## A4 · Install repeng — offline
🛠 **Add Input → Datasets** → add **repeng-offline-wheels** (built in Lesson 3). Then install from it with the internet **off**:

In [ ]:
# --no-index = never touch PyPI; --find-links points pip at the mounted wheels.
!pip install --no-index --find-links=/kaggle/input/repeng-offline-wheels repeng
import repeng
from repeng import ControlVector, ControlModel, DatasetEntry
print("repeng imported OK")

## A5 · Now pull the SAME model from Hugging Face — and watch it fail 💥
Deliberate. Mistral is **gated**: without an accepted license + a token, HF refuses the download. Run it and read the error.

In [ ]:
# 💥 EXPECTED TO FAIL — Mistral is gated, so this errors without an accepted
#     license + a token. Read the error message; that is the point of this cell.
from transformers import AutoModelForCausalLM
_ = AutoModelForCausalLM.from_pretrained(
    "mistralai/Mistral-7B-Instruct-v0.1", torch_dtype="auto")

## A6 · Fix it with Kaggle Secrets
🛠 (1) On huggingface.co, open the Mistral page and **accept the license**. (2) Create a token (Settings → Access Tokens). (3) In Kaggle: **Add-ons → Secrets**, add `HF_TOKEN` = your token and attach it. Then:

In [ ]:
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login
login(token=UserSecretsClient().get_secret("HF_TOKEN"))
print("Logged in to HF — the gated download would now succeed.")
print("(For the rest of the lab we use the Kaggle Models copy, which is already local.)")

## A7 · Build a control vector with repeng
Load the model through repeng's `ControlModel`, pick a **theme**, and train. Code is provided — the point is that it runs.

> **Memory note:** Mistral-7B in fp16 is ~14 GB — tight on a single 16 GB T4. If you hit CUDA OOM, try **8-bit** (the “smaller Mistral”) or a **smaller model** — see the troubleshooting note in `kaggle_assignment.md`. (Using both T4s is a bonus; it's finicky on Kaggle.)

In [ ]:
import json, torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from repeng import ControlVector, ControlModel, DatasetEntry

# 🛠 same Kaggle Models path as A3:
MODEL_PATH = "/kaggle/input/mistral/pytorch/7b-instruct-v0.1/1"

tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
tokenizer.pad_token_id = 0
# Default: fp16 Mistral-7B on ONE GPU (~14 GB — usually fits a single 16 GB T4, but tight).
model = AutoModelForCausalLM.from_pretrained(MODEL_PATH, torch_dtype=torch.float16).to("cuda:0")
# --- If you hit CUDA OOM, replace the line above with ONE of these (see assignment troubleshooting): ---
#   (1) 8-bit Mistral (~7 GB, one T4; needs bitsandbytes) — the "smaller Mistral" option:
#       model = AutoModelForCausalLM.from_pretrained(MODEL_PATH, load_in_8bit=True, device_map={"": 0})
#   (2) a smaller ungated model, e.g. Qwen/Qwen2.5-1.5B-Instruct — see the assignment's "smaller model" note.
# (Using BOTH T4s via device_map="auto" is the Bonus exercise — it's finicky on Kaggle.)
model = ControlModel(model, list(range(-5, -18, -1)))
user_tag, asst_tag = "[INST]", "[/INST]"

# 🛠 pick ONE theme (add it as a Kaggle Dataset, or upload the lab's datasets/ folder):
THEME_PATH  = "/kaggle/input/repeng-lab-themes/playful_vs_serious.json"
SUFFIX_PATH = "/kaggle/input/repeng-lab-themes/all_truncated_outputs.json"

theme = json.load(open(THEME_PATH))
output_suffixes = json.load(open(SUFFIX_PATH))

# keep training fast for the lab: use a subset of the suffix corpus
truncated = [
    tokenizer.convert_tokens_to_string(tokens[:i])
    for tokens in (tokenizer.tokenize(s) for s in output_suffixes[:256])
    for i in range(1, len(tokens))
]

def make_dataset(template, positive_personas, negative_personas, suffixes):
    ds = []
    for suffix in suffixes:
        for p, n in zip(positive_personas, negative_personas):
            ds.append(DatasetEntry(
                positive=f"{user_tag} {template.format(persona=p)} {asst_tag} {suffix}",
                negative=f"{user_tag} {template.format(persona=n)} {asst_tag} {suffix}"))
    return ds

dataset = make_dataset(theme["template"], theme["positive_personas"],
                       theme["negative_personas"], truncated)
model.reset()
vector = ControlVector.train(model, tokenizer, dataset)
print("Trained a control vector for theme:", theme["theme"])

In [ ]:
def generate_with_vector(prompt, vector, coeffs=(1.5, -1.5), max_new_tokens=100):
    pos, neg = coeffs
    if user_tag not in prompt:
        prompt = f"{user_tag} {prompt.strip()} {asst_tag}"
    ids = tokenizer(prompt, return_tensors="pt").to(model.device)
    st = dict(pad_token_id=tokenizer.eos_token_id, do_sample=False,
              max_new_tokens=max_new_tokens, repetition_penalty=1.1)
    model.reset()
    print("== baseline ==\n", tokenizer.decode(model.generate(**ids, **st)[0], skip_special_tokens=True), "\n")
    model.set_control(vector, pos)
    print("== + control ==\n", tokenizer.decode(model.generate(**ids, **st)[0], skip_special_tokens=True), "\n")
    model.set_control(vector, neg)
    print("== - control ==\n", tokenizer.decode(model.generate(**ids, **st)[0], skip_special_tokens=True))
    model.reset()

generate_with_vector(theme.get("suggested_prompt", "Give me a one-sentence pitch for a TV show."), vector)

## A8 · Save your output
Export the trained vector to `/kaggle/working/`. Then 🛠 **Save Version → Save & Run All (Commit)**; the file appears under the notebook's **Output** tab.

In [ ]:
# export_gguf uses the `gguf` library — the second wheel in the offline set.
vector.export_gguf("/kaggle/working/control_vector.gguf")
import os
print("saved:", [f for f in os.listdir("/kaggle/working") if f.endswith(".gguf")])

## A9 · Import vs. fork
You **forked** this notebook. You can also **import** one: 🛠 **File → Import Notebook** and upload `lab1a_allinone.ipynb` from the lab folder — same content, a different way in.

---
**Done!** Now answer the **Overall Reflection** in `kaggle_assignment.md`.